# PDF Extraction Agent
## Unstructured to Structured — Extracting Data from PDFs
⏱ ~12 min

**PDF documents are the most common format for unstructured knowledge** — research papers, contracts, reports, invoices, technical manuals. LLMs can read them, but first you need to get the text out in a form the model can actually use.

This workshop teaches you to build a **LangGraph agent** that downloads a PDF, extracts its text, and uses `with_structured_output()` to parse it into a typed Pydantic model — with automatic retry on failure.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why PDF extraction is hard; tool comparison |
| 2 | **Text Extraction** — pdfplumber hands-on: pages, tables, layout |
| 3 | **Structured Output** — `with_structured_output()` + Pydantic schemas |
| 4 | **LangGraph Agent** — Download → Extract → Retry loop |
| 5 | **Richer Schemas** — Tables, entities, nested objects |
| 6 | **Multiple Documents** — Parallel extraction with ThreadPoolExecutor |
| 7 | **Debugging & Evaluation** — What breaks and how to fix it |
| ★ | **Extensions** — OCR fallback, confidence scores, streaming |

---

### Prerequisites
- Python 3.10+ (a `.venv` with the requirements already installed)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)
- Internet access to download PDFs from arXiv (or bring your own)

### Key References
> Borchmann, L. et al. (2021). *Due: The Enterprise Document Understanding Evaluation Benchmark.* https://arxiv.org/abs/2111.15516  
> Mathew, M. et al. (2021). *DocVQA: A Dataset for VQA on Document Images.* WACV 2021. https://arxiv.org/abs/2007.00398  
> Vaswani, A. et al. (2017). *Attention Is All You Need.* NeurIPS 2017. https://arxiv.org/abs/1706.03762 (used as demo PDF)  
> pdfplumber documentation: https://github.com/jsvine/pdfplumber  
> LangGraph documentation: https://langchain-ai.github.io/langgraph/

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "langchain",
            "langchain-openai",
            "langgraph",
            "pdfplumber",
            "pypdf",
            "requests",
            "python-dotenv",
            "pydantic",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import io
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Literal, Optional, TypedDict

import requests

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

# ── Core imports ──────────────────────────────────────────────────────────────
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph
from pydantic import BaseModel, Field

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
key_ok = bool(key) and key.startswith("sk-")
print(f"OPENAI_API_KEY present and valid: {key_ok}")
if not key_ok:
    print()
    print("  ACTION REQUIRED — add your key before running LLM cells.")
    print("  Local: echo 'OPENAI_API_KEY=sk-...' >> .env")
    print("  Colab: Secrets panel → Add secret 'OPENAI_API_KEY'")

---

## Part 1 — Why PDF Extraction Is Hard

---

### The Problem

PDFs are **not a text format** — they are a page description language (PostScript descendant) that tells a renderer where to draw each character on screen. There is no concept of paragraphs, reading order, or table structure in the raw format. Everything you see as "text" must be reverse-engineered from character positions.

Three categories of PDFs exist, requiring different extraction strategies:

| PDF Category | What's inside | Extraction approach |
|---|---|---|
| **Text PDF** | Real Unicode characters embedded in the file | pdfplumber / pypdf — fast, accurate |
| **Scanned PDF** | Rasterized image of a page (no selectable text) | OCR (Tesseract / AWS Textract) |
| **Hybrid PDF** | Mix of text and image pages | Detect per-page; apply both strategies |

### Extraction Tool Comparison

| Library | Text accuracy | Tables | Images | Speed | License | Notes |
|---|---|---|---|---|---|---|
| **pdfplumber** | Excellent | Native table detection | No | Fast | MIT | Best for layout-sensitive docs |
| **pypdf** | Good | No | No | Very fast | BSD | Minimal deps; good for plain text |
| **PyPDF2** | Good | No | No | Fast | BSD | Legacy — use pypdf instead |
| **pdfminer.six** | Excellent | Via layout API | No | Medium | MIT | pdfplumber is built on top of this |
| **Docling** (IBM) | Excellent | Excellent | Via OCR | Slow | Apache 2.0 | State of the art; large model |
| **LlamaParse** | Excellent | Excellent | Via cloud | Medium | Paid API | Cloud-based; handles scans |
| **Tesseract** | Varies | No | Required | Slow | Apache 2.0 | OCR engine for scanned PDFs |

**This workshop uses `pdfplumber`** — zero cloud dependency, native table detection, and MIT-licensed.

### The Structured Extraction Pipeline

```
PDF file (bytes)
      │
      ▼
┌─────────────────┐
│  Text Extraction │  pdfplumber reads character positions
│  (pdfplumber)   │  and reconstructs readable text + tables
└────────┬────────┘
         │  raw text string
         ▼
┌─────────────────┐
│    Chunking     │  (optional) split long docs into
│  (if needed)    │  manageable pieces for the LLM context
└────────┬────────┘
         │  text chunk(s)
         ▼
┌──────────────────────────────────┐
│  LLM + with_structured_output()  │  function-calling guarantees
│       (Pydantic schema)          │  schema is satisfied
└────────┬─────────────────────────┘
         │  validated Pydantic object
         ▼
┌─────────────────┐
│  Structured     │  dict / JSON / database row
│  Output         │  ready for downstream processing
└─────────────────┘
```

> The key insight: the LLM acts as an **intelligent parser** — not a generator. You give it messy text; it returns a guaranteed-valid typed object. `with_structured_output()` achieves this via function-calling / JSON mode under the hood.

In [ ]:
# ─── 1-A: Prove that PDFs are NOT plain text ──────────────────────────────────
# Download the "Attention is All You Need" paper from arXiv
# and inspect the raw bytes — total binary gibberish without a parser.

PDF_URL = "https://arxiv.org/pdf/1706.03762"

print(f"Downloading: {PDF_URL}")
resp = requests.get(PDF_URL, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
resp.raise_for_status()
raw_bytes = resp.content

print(f"Downloaded:  {len(raw_bytes):,} bytes")
print(f"PDF magic number (first 8 bytes): {raw_bytes[:8]!r}")
print()
print("A 200-byte slice from the middle of the file (raw):")
print(repr(raw_bytes[5000:5200]))
print()
print("Observation: binary data — you need a parser to recover readable text.")

---

## Part 2 — Text Extraction with pdfplumber

---

### How pdfplumber Works

pdfplumber is built on `pdfminer.six`. It reads the PDF's internal character stream, computes the (x, y) position of every character, and reassembles them into lines and paragraphs based on proximity. Its **table detection algorithm** identifies grid structures by looking for horizontal and vertical rules (lines drawn in the PDF).

### Key API Surface

| Method | Returns | Use for |
|---|---|---|
| `pdf.pages` | `list[Page]` | Iterate over all pages |
| `page.extract_text()` | `str` or `None` | Full page text |
| `page.extract_text(x_tolerance=3)` | `str` | Fine-tune word spacing |
| `page.extract_tables()` | `list[list[list]]` | All tables on the page |
| `page.extract_table()` | `list[list]` | First (largest) table |
| `page.chars` | `list[dict]` | Every character with x, y, font, size |
| `page.words` | `list[dict]` | Words with bounding boxes |
| `page.crop((x0,y0,x1,y1))` | `Page` | Region of interest |
| `pdf.metadata` | `dict` | Title, author, creation date |

In [ ]:
# ─── 2-A: Basic text extraction — page by page ────────────────────────────────
import pdfplumber

with pdfplumber.open(io.BytesIO(raw_bytes)) as pdf:
    print(f"Total pages: {len(pdf.pages)}")
    print(f"Metadata:    {pdf.metadata}")
    print()

    for page_num in range(min(3, len(pdf.pages))):
        page = pdf.pages[page_num]
        text = page.extract_text() or ""
        # Collapse newlines for compact preview
        preview = text[:300].replace("\n", " / ")
        print(f"--- Page {page_num + 1} ({len(text)} chars) ---")
        print(f"  {preview}")
        print()

In [ ]:
# ─── 2-B: Concatenate all pages and inspect the full document ─────────────────
# This is how you produce a single string for the LLM.

with pdfplumber.open(io.BytesIO(raw_bytes)) as pdf:
    # Join pages with a blank line — preserves page boundary cues
    full_text = "\n\n".join(
        page.extract_text() or ""
        for page in pdf.pages
    )

print(f"Total extracted characters: {len(full_text):,}")
print(f"Total lines:                {full_text.count(chr(10)):,}")
print(f"Estimated tokens:           ~{len(full_text) // 4:,}  (4-char/token heuristic)")
print()
print("=== FIRST 800 CHARACTERS ===")
print(full_text[:800])

In [ ]:
# ─── 2-C: Table extraction ────────────────────────────────────────────────────
# pdfplumber detects tables from ruling lines drawn in the PDF.
# Academic papers typically have results tables — let's find them.

tables_found = []

with pdfplumber.open(io.BytesIO(raw_bytes)) as pdf:
    for page_num, page in enumerate(pdf.pages):
        tables = page.extract_tables()
        for tbl_idx, table in enumerate(tables):
            tables_found.append({
                "page": page_num + 1,
                "index": tbl_idx,
                "rows": len(table),
                "cols": max(len(row) for row in table) if table else 0,
                "data": table,
            })

print(f"Tables found across all pages: {len(tables_found)}")
print()

for t in tables_found[:3]:
    print(f"Page {t['page']}, table {t['index']+1}: {t['rows']} rows × {t['cols']} cols")
    for row in t["data"][:4]:  # first 4 rows
        cleaned = [
            str(cell).strip().replace("\n", " ") if cell else ""
            for cell in row
        ]
        print(f"  {' | '.join(cleaned[:5])}")
    print()

### Exercise 1 — Extract from Your Own PDF

Change `MY_PDF_URL` to any PDF you are interested in (a research paper, corporate report, whitepaper). Observe:

1. How many pages? How many characters?
2. Does `extract_text()` return readable text or garbled output?
3. Are any tables detected?

If the text is empty or garbled, the PDF is likely scanned — note it. (OCR fallback is covered in Part 8.)

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────
MY_PDF_URL = "https://arxiv.org/pdf/1706.03762"  # TODO: replace with your PDF URL

resp_ex1 = requests.get(MY_PDF_URL, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
resp_ex1.raise_for_status()

with pdfplumber.open(io.BytesIO(resp_ex1.content)) as pdf_ex1:
    num_pages = len(pdf_ex1.pages)
    # TODO: extract text from first 2 pages and print first 500 chars
    # TODO: check for tables on page 1
    text_ex1 = "\n".join(p.extract_text() or "" for p in pdf_ex1.pages[:2])

print(f"Pages: {num_pages}")
print(f"Characters extracted (first 2 pages): {len(text_ex1):,}")
print("\nFirst 500 chars:")
print(text_ex1[:500])

---

## Part 3 — Structured Output: Making the LLM Return a Schema

---

### The Problem with Freeform LLM Parsing

If you ask an LLM "extract the title and year from this paper, return JSON", you get:
- Sometimes valid JSON, sometimes wrapped in markdown code fences
- Sometimes extra fields, sometimes missing required fields
- Integers that come back as strings (`"2017"` instead of `2017`)
- No validation — you write your own parser and error handling

`with_structured_output(PydanticModel)` solves all of this by using **OpenAI's function-calling** interface under the hood. The LLM is forced to call a virtual "function" whose arguments must match your schema exactly.

### Three Approaches Compared

| Approach | Reliability | Validation | Type safety | Effort |
|---|---|---|---|---|
| `"Return JSON with fields..."` | Low — fragile | None | No | Low |
| `response_format={"type": "json_object"}` | Medium — JSON guaranteed, schema not | Partial | No | Low |
| `with_structured_output(PydanticModel)` | High — schema enforced | Full Pydantic | Yes | Low |

**Use `with_structured_output()` for all production extraction pipelines.**

### Pydantic Schema Design Rules

```
Field type rules:

  str         → any text field (title, abstract, key finding)
  int         → year, count, rank  (never str for numbers you compute with)
  float       → metric, score, percentage
  list[str]   → authors, keywords, bullet points
  Optional[X] → field that may not exist in every document
  Literal[..] → constrained vocabulary (document type, status)

Field description rules:

  - Be specific about format:  "Year as a 4-digit integer"
  - Name the fallback:         "If not found, return 'Unknown'"
  - Constrain the vocabulary:  "One of: paper/report/whitepaper"
```

In [ ]:
# ─── 3-A: Define a Pydantic schema and wire up with_structured_output ──────────

class PaperSchema(BaseModel):
    """Structured fields extracted from an academic paper."""
    title: str = Field(description="Full title of the paper")
    year: int = Field(description="Publication year as a 4-digit integer")
    key_contribution: str = Field(
        description="One sentence describing the main contribution of the paper"
    )
    architecture_name: str = Field(
        description=(
            "Name of the proposed model or architecture (e.g. Transformer, BERT). "
            "Return 'N/A' if no architecture is proposed."
        )
    )


# temperature=0 for deterministic extraction — higher values risk hallucinated field values
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
# with_structured_output wraps the model call so it returns a validated PaperSchema instance,
# not a raw string — internally it adds tool/function-call instructions to the prompt
extractor = llm.with_structured_output(PaperSchema)

print("Schema fields:")
for name, info in PaperSchema.model_fields.items():
    print(f"  {name}: {info.annotation.__name__}")
    print(f"    → {info.description}")
print()
print("Extractor type:", type(extractor).__name__)

In [ ]:
# ─── 3-B: Run structured extraction on real PDF text ──────────────────────────
# Use the first 3000 chars — enough to capture abstract and introduction.

# 3000 chars ≈ 750 tokens — captures the abstract + intro without expensive full-doc cost
# Alternative: send the full text but watch cost; gpt-4o-mini is cheap but latency scales
CHARS_LIMIT = 3000
text_for_extraction = full_text[:CHARS_LIMIT]

prompt = (
    f"Extract structured information from this academic paper text:\n\n"
    f"{text_for_extraction}\n\n"
    "Fill in all required fields. Use 'N/A' for fields not found in the text."
)

result: PaperSchema = extractor.invoke([HumanMessage(content=prompt)])

print("Extraction result (PaperSchema):")
print(f"  title:             {result.title}")
print(f"  year:              {result.year}  (type: {type(result.year).__name__})")
print(f"  key_contribution:  {result.key_contribution}")
print(f"  architecture_name: {result.architecture_name}")
print()
print("As JSON dict:")
print(json.dumps(result.model_dump(), indent=2))

In [ ]:
# ─── 3-C: Schema with Optional fields and lists ───────────────────────────────
# Real documents don't always have every field.
# Optional[T] lets the LLM return None when a field is absent.

class RichPaperSchema(BaseModel):
    """Extended schema for richer academic paper extraction."""
    title: str = Field(description="Full title of the paper")
    year: int = Field(description="Publication year as a 4-digit integer")
    authors: list[str] = Field(
        description="List of author names. Empty list if not found.",
        default_factory=list,
    )
    abstract: Optional[str] = Field(
        default=None,
        description="The abstract text, or null if not present in the excerpt",
    )
    key_contribution: str = Field(
        description="One sentence describing the main contribution"
    )
    architecture_name: str = Field(
        description="Name of proposed model or architecture. 'N/A' if none."
    )
    keywords: list[str] = Field(
        description="3-5 relevant keywords from the paper",
        default_factory=list,
    )


rich_extractor = llm.with_structured_output(RichPaperSchema)

rich_result: RichPaperSchema = rich_extractor.invoke(
    [HumanMessage(content=(
        f"Extract all fields from this paper text:\n\n{full_text[:4000]}\n\n"
        "Be thorough. For lists, return all items you can find."
    ))]
)

print("Rich extraction result:")
print(json.dumps(rich_result.model_dump(), indent=2))

### Exercise 2 — Design Your Own Schema

Design a `ContractSchema` for extracting structured fields from a legal contract or agreement PDF. Include at least:
- `parties: list[str]` — the contracting parties
- `effective_date: Optional[str]`
- `contract_type: str` — e.g. "NDA", "Service Agreement", "Employment"
- `key_obligations: list[str]` — main obligations for each party
- `termination_notice_days: Optional[int]`

**Challenge:** Add a `jurisdiction: Literal["US", "UK", "EU", "Other"]` field.

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────

class ContractSchema(BaseModel):
    """TODO: add field definitions below."""
    # TODO: parties: list[str]
    # TODO: effective_date: Optional[str]
    # TODO: contract_type: str
    # TODO: key_obligations: list[str]
    # TODO: termination_notice_days: Optional[int]
    # CHALLENGE: jurisdiction: Literal["US", "UK", "EU", "Other"]
    pass

# After defining your schema, create the extractor:
# contract_extractor = llm.with_structured_output(ContractSchema)
print("Define ContractSchema above, then uncomment the extractor line.")

---

## Part 4 — The LangGraph Extraction Agent

---

### Why an Agent Instead of a Direct Call?

A single LLM call works fine for short, clean PDFs. But in production:
- The LLM may fail to fill required fields (bad text quality, truncation)
- The PDF download may time out and need retrying
- You want automatic recovery from transient failures without caller-side retry logic

A **LangGraph agent** adds a retry loop with state tracking — failures are handled automatically inside the graph.

### The Graph

```
START
  │
  ▼
┌──────────┐
│ download │  fetch PDF bytes → extract text → truncate to CHARS_LIMIT
└────┬─────┘
     │  pdf_text stored in state
     ▼
┌──────────┐  ◄─────────────────────────────────────────────┐
│ extract  │  call LLM.with_structured_output(PaperSchema)   │
└────┬─────┘                                                  │
     │                                                        │ retry
     ▼                                                        │ (retries < MAX_RETRIES)
should_retry() ──── success? ──► END                         │
                                                              │
                 ──── failed? ────────────────────────────────┘
```

### State Machine Design

| State field | Type | Purpose |
|---|---|---|
| `url` | `str` | PDF source URL |
| `pdf_text` | `str` | Extracted text (populated by `download` node) |
| `extracted` | `dict` | Parsed fields (populated by `extract` node) |
| `retries` | `int` | Counter — prevents infinite retry loop |
| `success` | `bool` | True when extraction passes Pydantic validation |

**Why track `retries` in state?** The graph has no external memory between node calls. Every piece of information needed for routing must live in the state dict.

In [ ]:
# ─── 4-A: Define state and the download helper ────────────────────────────────

MAX_RETRIES = 3


# TypedDict gives LangGraph typed access to state keys without runtime overhead —
# each node receives this dict and returns a partial dict with only the keys it updates
class ExtractionState(TypedDict):
    url: str
    pdf_text: str
    extracted: dict
    retries: int
    success: bool


def download_pdf_text(url: str) -> str:
    """Download a PDF from URL and return extracted text string."""
    try:
        response = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
        response.raise_for_status()
        with pdfplumber.open(io.BytesIO(response.content)) as pdf:
            pages_text = [page.extract_text() or "" for page in pdf.pages[:4]]
            return " ".join(pages_text)
    except Exception as e:
        return f"[PDF download failed: {e}]"


print("ExtractionState and download_pdf_text defined.")

In [ ]:
# ─── 4-B: Define the graph nodes ──────────────────────────────────────────────

agent_extractor = llm.with_structured_output(PaperSchema)


def download_node(state: ExtractionState) -> dict:
    """Node 1: Download PDF and extract raw text.

    Returns an empty dict if pdf_text is already populated —
    important for retry loops so we don't re-download on each attempt.
    """
    if state["pdf_text"]:
        return {}  # already downloaded; skip
    text = download_pdf_text(state["url"])
    print(f"  [download] {len(text):,} chars from {state['url']}")
    return {"pdf_text": text[:CHARS_LIMIT]}


def extract_node(state: ExtractionState) -> dict:
    """Node 2: Use LLM to extract structured fields from the PDF text."""
    prompt = (
        f"Extract structured information from this academic paper text:\n\n"
        f"{state['pdf_text']}\n\n"
        "Fill in: title, year (integer), key_contribution, architecture_name. "
        "Use 'N/A' for any field not found in the text."
    )
    try:
        result = agent_extractor.invoke([HumanMessage(content=prompt)])
        print(f"  [extract] Success on attempt {state['retries'] + 1}")
        return {"extracted": result.model_dump(), "success": True}
    except Exception as e:
        print(f"  [extract] Failed (attempt {state['retries'] + 1}): {e}")
        return {
            "extracted": {},
            "success": False,
            "retries": state["retries"] + 1,
        }


# add_conditional_edges maps should_retry's return value to the next node name;
# returning END is a sentinel that exits the graph rather than routing to another node
def should_retry(state: ExtractionState) -> str:
    """Conditional edge: route back to extract on failure, else END."""
    if state["success"]:
        print("  [router] Extraction succeeded → END")
        return END
    if state["retries"] < MAX_RETRIES:
        print(f"  [router] Retry {state['retries']}/{MAX_RETRIES} → extract")
        return "extract"
    print(f"  [router] Max retries reached → END")
    return END


print("Nodes defined: download_node, extract_node, should_retry.")

In [ ]:
# ─── 4-C: Assemble and run the graph ──────────────────────────────────────────

def create_workflow():
    """Build and compile the PDF extraction LangGraph agent."""
    graph = StateGraph(ExtractionState)

    graph.add_node("download", download_node)
    graph.add_node("extract", extract_node)

    graph.set_entry_point("download")
    graph.add_edge("download", "extract")
    graph.add_conditional_edges(
        "extract",
        should_retry,
        {"extract": "extract", END: END},
    )

    # compile() locks the graph topology — after this, nodes and edges cannot be changed
    return graph.compile()


app = create_workflow()
# invoke() runs synchronously; use astream() if you want streaming updates from each node

print(f"Running extraction from: {PDF_URL}")
print()

final_state = app.invoke({
    "url": PDF_URL,
    "pdf_text": "",
    "extracted": {},
    "retries": 0,
    "success": False,
})

print()
if final_state["success"]:
    print("RESULT (success):")
    for k, v in final_state["extracted"].items():
        print(f"  {k}: {v}")
    print(f"  retries used: {final_state['retries']}")
else:
    print(f"RESULT: failed after {final_state['retries']} retries")

---

## Part 5 — Richer Schemas: Tables, Entities, and Nested Objects

---

So far we extracted flat fields. Real documents contain nested structures: tables with rows, entities with relationships, and sections with subsections.

### When to Use Nested Schemas

```
Flat schema (good for single-value fields):

  PaperSchema.title:             str  ← one value per document
  PaperSchema.year:              int  ← one value per document

Nested schema (needed for repeating groups):

  PaperSchema.results: list[ExperimentResult]
    ExperimentResult.metric:  str
    ExperimentResult.value:   float
    ExperimentResult.dataset: str

Why nest?
  - Avoids name collision (result_1_metric, result_1_value, result_2_...)
  - Pydantic validates each sub-model individually
  - Maps naturally to database rows or JSON arrays
```

In [ ]:
# ─── 5-A: Nested schema for paper results tables ──────────────────────────────

class ExperimentResult(BaseModel):
    """One row from the paper's results / experiments table."""
    metric: str = Field(description="Metric name, e.g. BLEU score, accuracy, perplexity")
    value: str = Field(
        description="Reported value (keep as string to preserve units like '28.4 BLEU')"
    )
    dataset: str = Field(
        description="Dataset or benchmark name. 'Unknown' if not specified."
    )
    model_variant: str = Field(
        description="Model variant reported, e.g. 'Transformer (big)'. 'N/A' if single model."
    )


class DetailedPaperSchema(BaseModel):
    """Full structured extraction of an academic paper."""
    title: str = Field(description="Full title")
    year: int = Field(description="4-digit publication year")
    authors: list[str] = Field(
        description="List of author names",
        default_factory=list,
    )
    key_contribution: str = Field(description="Main contribution in one sentence")
    architecture_name: str = Field(description="Proposed model name. 'N/A' if none.")
    results: list[ExperimentResult] = Field(
        description="Key experimental results. Empty list if no results table found.",
        default_factory=list,
    )
    limitations: Optional[str] = Field(
        default=None,
        description="Stated limitations of the approach, or null if not mentioned",
    )


detailed_extractor = llm.with_structured_output(DetailedPaperSchema)

# Use more text to reach results tables (usually mid-paper)
detailed_result: DetailedPaperSchema = detailed_extractor.invoke(
    [HumanMessage(content=(
        f"Extract all fields. For results, look for BLEU scores, accuracy, "
        f"or other metrics in tables.\n\n{full_text[:6000]}"
    ))]
)

print(json.dumps(detailed_result.model_dump(), indent=2))

In [ ]:
# ─── 5-B: Use pdfplumber table output directly as structured input ─────────────
# Feeding explicit table text to the LLM (rather than whole-page prose)
# often improves extraction accuracy for tabular data.

def extract_tables_as_text(pdf_bytes: bytes) -> str:
    """Extract all tables from a PDF and format them as readable text blocks."""
    blocks = []
    with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
        for page_num, page in enumerate(pdf.pages):
            for tbl_idx, table in enumerate(page.extract_tables()):
                if not table:
                    continue
                header = " | ".join(
                    str(c).strip().replace("\n", " ") if c else ""
                    for c in table[0]
                )
                rows = [
                    " | ".join(
                        str(c).strip().replace("\n", " ") if c else ""
                        for c in row
                    )
                    for row in table[1:]
                ]
                blocks.append(
                    f"[TABLE page {page_num+1}, #{tbl_idx+1}]\n"
                    + header + "\n"
                    + "\n".join(rows)
                )
    return "\n\n".join(blocks) if blocks else "No tables found."


table_text = extract_tables_as_text(raw_bytes)
print(f"Table text extracted: {len(table_text)} chars")
print()
print(table_text[:1000] if table_text != "No tables found." else "No tables found in this PDF.")

---

## Part 6 — Processing Multiple Documents in Parallel

---

A single-document pipeline becomes a **batch extraction system** with `ThreadPoolExecutor`. Each PDF runs in its own thread — the bottleneck is network I/O and LLM latency, both of which benefit from concurrency.

### Parallelism Options

| Approach | Best for | Notes |
|---|---|---|
| `ThreadPoolExecutor` | I/O-bound tasks (HTTP, disk) | Simple; works with sync LangChain calls |
| `asyncio` + `ainvoke` | High-concurrency servers | Requires async-compatible code throughout |
| LangGraph `batch()` | Multiple inputs to same graph | Native batching; auto-parallelises |

In [ ]:
# ─── 6-A: Batch extraction with ThreadPoolExecutor ────────────────────────────

PAPER_URLS = [
    ("Attention Is All You Need", "https://arxiv.org/pdf/1706.03762"),
    ("BERT",                     "https://arxiv.org/pdf/1810.04805"),
    ("GPT-3",                    "https://arxiv.org/pdf/2005.14165"),
]


def extract_one(name_url: tuple) -> dict:
    """Process one (name, url) pair and return a result dict."""
    name, url = name_url
    state = app.invoke({
        "url": url,
        "pdf_text": "",
        "extracted": {},
        "retries": 0,
        "success": False,
    })
    return {"name": name, "url": url, "state": state}


print(f"Processing {len(PAPER_URLS)} papers in parallel...")
print("(Expect ~15-25 seconds — network + LLM latency, all concurrent)")
print()

t0 = time.time()
batch_results = []

# ThreadPoolExecutor works here because the bottleneck is I/O (network + LLM API),
# not CPU — for CPU-bound tasks use ProcessPoolExecutor instead
with ThreadPoolExecutor(max_workers=3) as executor:
    futures = {executor.submit(extract_one, item): item for item in PAPER_URLS}
    for future in as_completed(futures):
        r = future.result()
        batch_results.append(r)
        status = "OK" if r["state"]["success"] else "FAILED"
        title = r["state"]["extracted"].get("title", "N/A")[:55]
        print(f"  [{status}] {r['name']}: {title}")

elapsed = time.time() - t0
print(f"\nCompleted in {elapsed:.1f}s — sequential estimate: ~{elapsed * len(PAPER_URLS):.0f}s")

In [ ]:
# ─── 6-B: Summarise batch results as a comparison table ───────────────────────

print(f"{'Paper':<40} {'Year':<6} {'Architecture':<28} {'OK?'}")
print("-" * 82)

for item in batch_results:
    state = item["state"]
    if state["success"]:
        ex = state["extracted"]
        print(
            f"{ex.get('title', 'N/A')[:38]:<40} "
            f"{str(ex.get('year', '?')):<6} "
            f"{ex.get('architecture_name', 'N/A')[:26]:<28} "
            f"Yes"
        )
    else:
        print(
            f"{item['name']:<40} {'?':<6} {'N/A':<28} "
            f"No (retries={state['retries']})"
        )

---

## Part 7 — Debugging and Evaluation

---

### Common Failure Modes

| Failure | Symptom | Root cause | Fix |
|---|---|---|---|
| **Blank text** | `pdf_text` is empty or `[PDF download failed]` | Scanned PDF or network error | OCR fallback (Part 8) |
| **Truncated extraction** | Fields missing or abbreviated | `CHARS_LIMIT` too small | Increase limit or chunk |
| **Wrong year** | Returns 9999 or a citation year | Year in header/footer confused | Tighten field description |
| **Schema validation fails** | `with_structured_output` raises | LLM returned wrong type | Add type coercion or increase retries |
| **Hallucinated fields** | Fields filled with confident nonsense | LLM inventing when text is poor | Prompt: "Return 'N/A' if not found" |
| **Infinite retry** | Agent loops forever | Missing retries guard | Always check `retries < MAX_RETRIES` |

### Debugging Checklist

1. **Print the extracted text** — is it readable? Is it the right document?
2. **Print the full prompt** — what exactly is the LLM receiving?
3. **Inspect the raw LLM response** — does the schema mismatch happen at the type level?
4. **Check `retries`** in the final state — how many attempts were needed?

In [ ]:
# ─── 7-A: Diagnostic — print the exact prompt the LLM receives ────────────────
# The #1 debugging step: seeing the full prompt reveals most extraction problems.

sample_text = full_text[:CHARS_LIMIT]
extraction_prompt = (
    f"Extract structured information from this academic paper text:\n\n"
    f"{sample_text}\n\n"
    "Fill in: title, year (integer), key_contribution, architecture_name. "
    "Use 'N/A' for any field not found in the text."
)

print("=== EXTRACTION PROMPT SENT TO LLM ===")
print(f"Total: {len(extraction_prompt)} chars (~{len(extraction_prompt) // 4:,} tokens)")
print()
print(extraction_prompt[:1200])
if len(extraction_prompt) > 1200:
    print(f"... [{len(extraction_prompt) - 1200} more chars]")

In [ ]:
# ─── 7-B: Experiment — effect of CHARS_LIMIT on extraction quality ─────────────
# More text is not always better: diminishing returns, higher cost.

def extract_with_limit(text: str, chars_limit: int) -> dict:
    truncated = text[:chars_limit]
    prompt = (
        f"Extract structured fields from this paper text:\n\n{truncated}\n\n"
        "Fill in: title, year (integer), key_contribution, architecture_name. "
        "Use 'N/A' if not found."
    )
    try:
        r = extractor.invoke([HumanMessage(content=prompt)])
        return {"limit": chars_limit, "success": True, **r.model_dump()}
    except Exception as e:
        return {"limit": chars_limit, "success": False, "error": str(e)}


print(f"{'Limit':>6}  {'Title found':<12}  {'Year':>6}  {'Architecture':<30}  {'Tokens est.':>12}")
print("-" * 80)

for limit in [500, 1000, 2000, 4000]:
    r = extract_with_limit(full_text, limit)
    if r["success"]:
        found = "Yes" if r["title"] not in ("N/A", "") else "No"
        print(
            f"{limit:>6}  {found:<12}  {r['year']:>6}  "
            f"{r['architecture_name'][:28]:<30}  {limit // 4:>12,}"
        )
    else:
        print(f"{limit:>6}  {'FAILED':<12}  {'?':>6}  {r.get('error', '')[:28]:<30}")

### Exercise 3 — Build an Invoice Extraction Agent

An **invoice schema** is a different domain from academic papers. Invoices have:
- `invoice_number: str`
- `vendor_name: str`
- `invoice_date: str` (ISO format `YYYY-MM-DD`)
- `total_amount: float`
- `currency: str` (e.g. `"USD"`, `"EUR"`)
- `line_items: list[LineItem]` — each item has `description`, `quantity`, `unit_price`, `amount`

**Tasks:**
1. Define `LineItem` and `InvoiceSchema` as Pydantic models
2. Create `invoice_extractor = llm.with_structured_output(InvoiceSchema)`
3. Test it on the `SAMPLE_INVOICE` string below
4. Verify `total_amount` is a `float`, not a string
5. **Challenge:** wire it into the LangGraph agent from Part 4

In [ ]:
# ── Exercise 3 Starter ────────────────────────────────────────────────────────

class LineItem(BaseModel):
    # TODO: description: str
    # TODO: quantity: float
    # TODO: unit_price: float
    # TODO: amount: float
    pass


class InvoiceSchema(BaseModel):
    # TODO: invoice_number: str
    # TODO: vendor_name: str
    # TODO: invoice_date: str
    # TODO: total_amount: float
    # TODO: currency: str
    # TODO: line_items: list[LineItem]
    pass


SAMPLE_INVOICE = """
INVOICE #INV-2024-0042
Date: 2024-03-15
Vendor: Acme Software Ltd
Bill To: Example Corp

Items:
- Cloud Storage (500 GB) x 1  @ $49.00  =  $49.00
- API calls (1M requests) x 3 @ $12.00  =  $36.00
- Support plan (monthly)  x 1 @ $199.00 = $199.00

Subtotal: $284.00
Tax (8%): $22.72
Total: $306.72 USD
"""

# TODO: invoice_extractor = llm.with_structured_output(InvoiceSchema)
# TODO: invoice_result = invoice_extractor.invoke([HumanMessage(content=f"Extract invoice fields:\n\n{SAMPLE_INVOICE}")])
# TODO: print(json.dumps(invoice_result.model_dump(), indent=2))
# TODO: assert isinstance(invoice_result.total_amount, float), "total_amount must be float"

print("Define your schemas above, then uncomment and run the extractor.")

---

## Part 8 ★ — Extensions (Bonus)

---

### OCR Fallback for Scanned PDFs

When `page.extract_text()` returns `None` or empty, the page is a scanned image. Pattern:

```python
text = page.extract_text()
if not text:
    # Convert page to image at 300 DPI, then run OCR
    img = page.to_image(resolution=300).original
    text = pytesseract.image_to_string(img)  # local OCR
    # OR: use aws_textract_client.detect_document_text() for higher accuracy
```

### Confidence Scores

Add a `confidence: float` field (0.0–1.0) and instruct the LLM to self-report:

```python
confidence: float = Field(
    description="Confidence in extraction: 0.0 (guessing) to 1.0 (text unambiguous)",
    ge=0.0, le=1.0
)
```

Flag extractions with `confidence < 0.7` for human review.

### LangGraph Parallelism with `Send`

For multi-page documents, use LangGraph's `Send` primitive to fan out page-by-page processing in parallel, then merge results in a final node — graph-native parallelism without `ThreadPoolExecutor`:

```python
from langgraph.constants import Send

def route_pages(state):
    return [Send("extract_page", {"page_text": p}) for p in state["pages"]]
```

In [ ]:
# ─── 8-A: Detect scanned vs text pages ───────────────────────────────────────
# Simple heuristic: fewer than 50 chars on a page likely means it's scanned.

SCAN_THRESHOLD = 50  # chars below this = possibly scanned

with pdfplumber.open(io.BytesIO(raw_bytes)) as pdf:
    print(f"{'Page':<6} {'Chars':<8} {'Type':<12} {'Preview'}")
    print("-" * 70)
    for i, page in enumerate(pdf.pages):
        text = page.extract_text() or ""
        chars = len(text)
        page_type = "TEXT" if chars >= SCAN_THRESHOLD else "SCANNED?"
        preview = text[:40].replace("\n", " ") if text else "(empty)"
        print(f"{i+1:<6} {chars:<8} {page_type:<12} {preview}")

In [ ]:
# ─── 8-B: Add confidence score to the extraction schema ───────────────────────

class PaperSchemaWithConfidence(BaseModel):
    """Paper schema extended with a self-reported confidence score."""
    title: str = Field(description="Full title of the paper")
    year: int = Field(description="4-digit publication year")
    key_contribution: str = Field(description="Main contribution in one sentence")
    architecture_name: str = Field(description="Proposed model name. 'N/A' if none.")
    confidence: float = Field(
        description=(
            "Your confidence in this extraction from 0.0 (mostly guessing) "
            "to 1.0 (text was unambiguous and complete). "
            "Use 0.5 when some fields were inferred rather than found verbatim."
        ),
        ge=0.0,
        le=1.0,
    )


# The field description IS the instruction to the model — write it like a prompt, not a label
confident_extractor = llm.with_structured_output(PaperSchemaWithConfidence)

confident_result = confident_extractor.invoke(
    [HumanMessage(content=(
        f"Extract structured fields and report your confidence:\n\n"
        f"{full_text[:CHARS_LIMIT]}"
    ))]
)

print("Extraction with confidence score:")
print(json.dumps(confident_result.model_dump(), indent=2))
print()
flag = "REVIEW" if confident_result.confidence < 0.7 else "OK"
print(f"Quality gate: confidence={confident_result.confidence:.2f} → {flag}")

---

## What's Next?

You now have a complete PDF extraction pipeline. Here is where to go deeper:

### Build on this pattern
- **Example 13 — Structured Output** ([`../13-structured-output/`](../13-structured-output/)): deep dive into `with_structured_output()` without the PDF layer — covers all field types, validation, and error handling patterns.
- **Example 12 — Basic RAG** ([`../12-basic-rag-notebook/rag_workbook.ipynb`](../12-basic-rag-notebook/rag_workbook.ipynb)): once you have extracted text from PDFs, embed and retrieve it. The natural next step after this workshop.
- **Example 17 — Corrective RAG** ([`../17-corrective-rag/`](../17-corrective-rag/)): retry and self-correction patterns applied to RAG grading — same conditional-edge approach used here.
- **Example 30 — Agentic RAG** ([`../30-agentic-rag/`](../30-agentic-rag/)): full agentic loop over a document corpus.

### Go to production
- **OCR**: `pytesseract` (local, free) or AWS Textract / Google Document AI (cloud, higher accuracy on scans)
- **Docling** (IBM): state-of-the-art PDF parsing — tables, figures, reading order — outputs Markdown or JSON natively
- **LangSmith tracing**: set `LANGCHAIN_TRACING_V2=true` to see every graph step in a dashboard
- **Caching**: use `langchain_core.caches.InMemoryCache` or Redis to avoid re-extracting the same PDF twice

### Further reading
- Borchmann, L. et al. (2021). *Due: The Enterprise Document Understanding Evaluation Benchmark.* https://arxiv.org/abs/2111.15516
- Mathew, M. et al. (2021). *DocVQA: A Dataset for VQA on Document Images.* https://arxiv.org/abs/2007.00398
- pdfplumber GitHub: https://github.com/jsvine/pdfplumber
- LangGraph conditional edges: https://langchain-ai.github.io/langgraph/how-tos/branching/
- Pydantic v2 fields: https://docs.pydantic.dev/latest/concepts/fields/

---

*Workshop complete. Open Example 12 next — embed the text you just extracted and build a Q&A system over it.*

---

## Exercise Answer Key

Use this section after attempting the exercises yourself. These are reference solutions — not the only correct answers.

### Exercise 1 — Extract from Your Own PDF

**What to look for:**

- `extract_text()` returns `None` on scanned pages — always guard with `or ""`.
- Page count and character count vary wildly. A dense research paper packs ~15,000 chars into 15 pages; a scan-heavy annual report may have 200 pages but return almost no text.
- If tables are not detected, the PDF may use spaces to simulate grid layout rather than actual ruling lines — pdfplumber's table detector relies on visible PDF lines.

**Sample pattern for safe multi-page extraction:**

```python
with pdfplumber.open(io.BytesIO(resp.content)) as pdf:
    print(f"Pages: {len(pdf.pages)}")
    text_pages = []
    for i, page in enumerate(pdf.pages[:5]):
        t = page.extract_text() or ""          # never None
        tables = page.extract_tables()
        print(f"  Page {i+1}: {len(t)} chars, {len(tables)} table(s)")
        text_pages.append(t)
    full = "\n\n".join(text_pages)
```

### Exercise 2 — Design Your Own Schema

**Sample `ContractSchema` solution:**

In [ ]:
# Answer key — Exercise 2

class ContractSchemaAnswer(BaseModel):
    """Structured fields extracted from a legal contract."""
    parties: list[str] = Field(
        description="Names of all parties to the contract",
        default_factory=list,
    )
    effective_date: Optional[str] = Field(
        default=None,
        description="Effective date in YYYY-MM-DD format, or null if not stated",
    )
    contract_type: str = Field(
        description="Contract type: NDA, Service Agreement, Employment, License, or Other"
    )
    key_obligations: list[str] = Field(
        description="Main obligations for each party as individual bullet-point strings",
        default_factory=list,
    )
    termination_notice_days: Optional[int] = Field(
        default=None,
        description="Days notice required to terminate. null if not specified.",
    )
    jurisdiction: Literal["US", "UK", "EU", "Other"] = Field(
        description="Governing law jurisdiction. Use 'Other' if unclear."
    )


# Verify the schema instantiates correctly with sample data
sample = ContractSchemaAnswer(
    parties=["Acme Inc", "Widget Corp"],
    effective_date="2024-01-01",
    contract_type="NDA",
    key_obligations=[
        "Acme: keep all shared information confidential",
        "Widget: not to use information for competing products",
    ],
    termination_notice_days=30,
    jurisdiction="US",
)

print("ContractSchema is valid:")
print(json.dumps(sample.model_dump(), indent=2))

### Exercise 3 — Invoice Extraction Agent

**Key points:**

- `total_amount: float` — always use numeric types for values you might aggregate. Never `str`.
- `line_items: list[LineItem]` — nested Pydantic models validate each row individually.
- Prompt tip: tell the LLM "exclude tax and subtotal lines from line_items" to keep data clean.
- When the LLM cannot determine currency from a symbol alone, `"USD"` is a reasonable default if `$` is present.

In [ ]:
# Answer key — Exercise 3

class LineItemAnswer(BaseModel):
    description: str = Field(description="Product or service description")
    quantity: float = Field(description="Quantity ordered")
    unit_price: float = Field(description="Price per unit in the invoice currency")
    amount: float = Field(description="Line total (quantity × unit_price)")


class InvoiceSchemaAnswer(BaseModel):
    invoice_number: str = Field(description="Invoice identifier, e.g. INV-2024-0042")
    vendor_name: str = Field(description="Name of the vendor issuing the invoice")
    invoice_date: str = Field(
        description="Invoice date in YYYY-MM-DD format. 'Unknown' if not found."
    )
    total_amount: float = Field(description="Total amount due as a float number")
    currency: str = Field(description="Three-letter currency code, e.g. USD, EUR, GBP")
    line_items: list[LineItemAnswer] = Field(
        description=(
            "Individual invoice line items. "
            "Exclude tax lines and subtotal summary rows."
        ),
        default_factory=list,
    )


invoice_extractor_ans = llm.with_structured_output(InvoiceSchemaAnswer)

invoice_result = invoice_extractor_ans.invoke(
    [HumanMessage(content=f"Extract all invoice fields:\n\n{SAMPLE_INVOICE}")]
)

print("Invoice extraction result:")
print(json.dumps(invoice_result.model_dump(), indent=2))
print()
print(f"total_amount type: {type(invoice_result.total_amount).__name__} — expected: float")
print(f"line_items count:  {len(invoice_result.line_items)} — expected: 3")
assert isinstance(invoice_result.total_amount, float), "total_amount must be float"
assert len(invoice_result.line_items) == 3, "Expected 3 line items"
print("\nAll assertions passed.")